In [ ]:
import pandas as pd
from dataframe_modify.add_positions import add_positions
from dataframe_modify.swap_perspective import swap_y
from dataframe_modify.swap_perspective import generalize_perspective_sb360
from dataframe_modify.grid_areas import add_grid_areas
from sequences.get_sequences import get_sequences
from sequences.find_fast_actions import find_fast_actions_forward_perspective
from fast_actions.get_dataframe import get_fast_action_row

In [ ]:
euro_matches = pd.read_pickle('esa_21_22_matches.pkl')

In [ ]:
# euro_matches
euro_matches = add_positions(euro_matches)


In [ ]:
euro_matches, ids_to_drop = swap_y(euro_matches)
for id_to_drop in ids_to_drop:
    euro_matches = euro_matches[euro_matches['id'] != id_to_drop]
euro_matches = generalize_perspective_sb360(euro_matches)
euro_matches = add_grid_areas(euro_matches)
euro_sequences = get_sequences(euro_matches)

In [8]:
euro_sequences[0]


,id,index,period,timestamp,minute,second,type,possession,possession_team,play_pattern,...,context_locations,teammate_locations,actor_locations,start_x,start_y,end_x,end_y,start_area,end_area,area_code
0,60a052f2-babe-496d-b809-25ef0b0dbef0,5,1,00:00:01.494,0,1,Pass,2,Śląsk Wrocław,From Kick Off,...,[],[],[],60.0,40.0,48.0,35.8,F5,E5,F5E5
1,de193e98-0072-400b-b142-a4062976024e,6,1,00:00:03.243,0,3,Ball Receipt*,2,Śląsk Wrocław,From Kick Off,...,"[[41.982529937455325, 20.608110991001567], [38...","[True, True, True, False, True, True, False, F...","[False, False, True, False, False, False, Fals...",48.0,35.8,,,E5,,E5
2,dc6874ea-ce42-4a49-9c28-b8ebaced6f94,7,1,00:00:03.846,0,3,Pass,2,Śląsk Wrocław,From Kick Off,...,"[[39.827211598840655, 17.73565647944835], [36....","[True, True, True, True, False, False, False, ...","[False, False, True, False, False, False, Fals...",47.1,35.8,34.8,61.1,E5,D2,E5D2
3,d68fecbd-9a0d-4fb8-a960-48621169f300,8,1,00:00:05.699,0,5,Ball Receipt*,2,Śląsk Wrocław,From Kick Off,...,"[[36.823177251473425, 23.41056583071014], [33....","[True, True, True, False, True, True, False, F...","[False, False, True, False, False, False, Fals...",34.8,61.1,,,D2,,D2
4,1e1d5c9d-e2b8-4724-b361-401e72b35c7a,9,1,00:00:05.699,0,5,Carry,2,Śląsk Wrocław,From Kick Off,...,"[[36.823177251473425, 23.41056583071014], [33....","[True, True, True, False, True, True, False, F...","[False, False, True, False, False, False, Fals...",34.8,61.1,34.8,61.1,D2,D2,D2D2
5,f885b37b-6302-4009-b4cf-859986b3e430,10,1,00:00:06.536,0,6,Pass,2,Śląsk Wrocław,From Kick Off,...,"[[35.98731948691805, 23.632161258833108], [32....","[True, True, True, False, True, True, False, F...","[False, False, True, False, False, False, Fals...",34.8,61.1,43.7,79.0,D2,E1,D2E1
6,bd633f0a-6a07-4f86-b45d-96949242dc86,11,1,00:00:07.818,0,7,Ball Receipt*,2,Śląsk Wrocław,From Kick Off,...,"[[30.23254074583865, 48.43165727000731], [30.0...","[True, True, False, True, True, True, False, F...","[False, False, False, False, False, True, Fals...",43.7,79.0,,,E1,,E1
7,20e473e1-fafd-4f34-b33f-6399029601cd,12,1,00:00:07.818,0,7,Pass,2,Śląsk Wrocław,From Kick Off,...,"[[30.23254074583865, 48.83165724616545], [30.0...","[True, True, False, True, True, True, False, F...","[False, False, False, False, False, True, Fals...",43.7,79.4,24.7,74.8,E1,C1,E1C1
8,0e6111d7-0e89-4ca3-a96b-110d18569e13,13,1,00:00:09.965,0,9,Ball Receipt*,2,Śląsk Wrocław,From Kick Off,...,"[[28.660291457520742, 52.065658877333206], [33...","[True, False, True, False, True, False, False,...","[False, False, True, False, False, False, Fals...",24.7,74.8,,,C1,,C1
9,5471c4f6-57f0-4fb0-bc7d-e95175041dd1,14,1,00:00:09.965,0,9,Carry,2,Śląsk Wrocław,From Kick Off,...,"[[28.660291457520742, 52.065658877333206], [33...","[True, False, True, False, True, False, False,...","[False, False, True, False, False, False, Fals...",24.7,74.8,24.7,72.2,C1,C1,C1C1


In [9]:
matches_fast_actions = find_fast_actions_forward_perspective(sequences=euro_sequences)

In [16]:
matches_fast_actions[0]['obv_for_net'].values

array(['', 0.005242722, -0.009598072, ''], dtype=object)

In [20]:
matches_fast_actions[0]['obv_total_net'].values


array(['', 0.008214474, -0.021139149, ''], dtype=object)

In [ ]:
import math

def get_opponent_distances(seq):
    start_event_locations = [[x, y] for x, y in zip(seq['start_x'].values, seq['start_y'].values)]
    enemies_locations_sb_360 = []
    for context_locations, teammate_locations in zip(seq['context_locations'].values, seq['teammate_locations'].values):
        enemies_locations = [location if not teammate else '' for location, teammate 
                            in zip(context_locations, teammate_locations)]
        enemies_locations = list(filter(lambda a: a != '', enemies_locations))
        enemies_locations_sb_360.append(enemies_locations)
    enemies_distances = 0
    n = 0
    for event_loc, enemies_loc in zip(start_event_locations, enemies_locations_sb_360):
        for enemy_loc in enemies_loc:
            enemies_distances += math.dist(event_loc, [enemy_loc[0], enemy_loc[1]])
            n += 1
    return enemies_distances / n


print(get_opponent_distances(matches_fast_actions[0]))

In [ ]:
def get_key_pass_len(seq):
    team_half = {'A', 'B', 'C', 'D', 'E', 'F'}
    for event_id, event_type, end_area in zip(seq['id'].values, seq['type'].values, seq['end_area'].values):
        if event_type == 'Pass' and end_area[0] not in team_half:
            counterattack_pass = seq.loc[seq['id'] == event_id]
            start_x = counterattack_pass['start_x'].values[0]
            start_y = counterattack_pass['start_y'].values[0]
            end_x = counterattack_pass['end_x'].values[0]
            end_y = counterattack_pass['end_y'].values[0]
            return math.dist([start_x, start_y], [end_x, end_y])
    return 0

get_key_pass_len(matches_fast_actions[1])

In [ ]:
# new_df['start_x'] = [get_seq_start_loc(seq, 'start_x')]
#     new_df['start_y'] = [get_seq_start_loc(seq, 'start_y')]
#     new_df['end_x'] = [get_seq_end_loc(seq, 'x')]
#     new_df['end_y'] = [get_seq_end_loc(seq, 'y')]
#     new_df['key_pass_len'] = [get_key_pass_len(seq)]
#     new_df['opponent_distances'] = [get_opponent_distances(seq)]
#     new_df['obv'] = [get_fast_action_obv(seq)]
#     new_df['xg'] = [get_fast_action__xg(seq)]

In [11]:
from fast_actions.get_dataframe import get_seq_start_loc, get_seq_end_loc, get_fast_action_obv, get_fast_action__xg

seq = matches_fast_actions[0]

print(get_seq_start_loc(seq, 'start_x'))
print(get_seq_start_loc(seq, 'start_y'))
print(get_seq_end_loc(seq, 'x'))
print(get_seq_end_loc(seq, 'y'))
print(get_fast_action_obv(seq))
print(get_fast_action__xg(seq))

43.7
30.200000000000003
60.9
44.6
-0.004355349999999999
0
